# LeakCheck V2: Multi-Dataset EEG Leakage Auditing Benchmark Runner
### Zero-Cost Cloud Architecture (Google Colab & Persistent Cloud Storage)

**Author:** Joyjit Das (`JOYJIT-06`, ORCID: `0009-0008-0529-4104`)  
**Public Repository:** [https://github.com/JOYJIT-06/LeakCheck](https://github.com/JOYJIT-06/LeakCheck)  
**Branch:** `v2-dev`  

---

> **Zero-Cost Constraint Notice**  
> This notebook executes entirely on free Google Colab compute instances (Standard CPU or T4 GPU).  
> Bulk multi-gigabyte EEG datasets stream directly from public repositories (GigaScience, BNCI, NEMAR, PhysioNet, Zenodo) into Google Drive (`My Drive/LeakCheck_Cloud/datasets/`), completely bypassing local user machines.

### Supported Datasets (All 11 Registered Adapters):
1. `physionet` (PhysioNet EEGMMIDB, N=109/105)
2. `openbmi` (OpenBMI / Lee et al. 2019, N=54/34)
3. `cho2017` (Cho et al. 2017, N=52)
4. `bnci2014_001` (BCI Competition IV-2a, N=9)
5. `bnci2014_004` (BCI Competition IV-2b, N=9)
6. `high_gamma` (Schirrmeister et al. 2017, N=14)
7. `shin2017a` (Shin et al. 2017A, N=29)
8. `weibo2014` (Weibo et al. 2014, N=10)
9. `grosse_wentrup2009` (Grosse-Wentrup et al. 2009, N=10)
10. `zhou2016` (Zhou et al. 2016, N=4)
11. `yang2025` (Yang et al. 2025, N=60+)

## 1. Environment Setup and Repository Clone

In [ ]:
# Mount Google Drive for persistent zero-cost storage
from google.colab import drive
drive.mount('/content/drive')

# Clone the v2-dev development branch
!rm -rf LeakCheck
!git clone -b v2-dev https://github.com/JOYJIT-06/LeakCheck.git
%cd LeakCheck

# Install dependencies in editable mode
!pip install -q -e .
!pip install -q moabb

## 2. Configure Benchmark Target Dataset

In [ ]:
# Select target dataset adapter from the registered 11
DATASET = "cho2017"  # Change to any: physionet, openbmi, cho2017, bnci2014_001, bnci2014_004, etc.
DATA_DIR = f"/content/drive/MyDrive/LeakCheck_Cloud/datasets/{DATASET}"
OUT_DIR = f"/content/drive/MyDrive/LeakCheck_Cloud/benchmark_results/{DATASET}_v2"

import os
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

import emi
from emi.datasets_base import get_adapter, list_registered_adapters
import emi.datasets_adapters

adapter = get_adapter(DATASET, data_dir=DATA_DIR)
meta = adapter.metadata
print(f"Target Dataset: {meta.dataset_name}")
print(f"Citation: {meta.citation}")
print(f"Channels: {meta.channel_count} | Fs: {meta.sampling_rate} Hz | Planned Cohort: {meta.n_subjects}")

## 3. Direct Cloud Acquisition / Streaming
Downloads raw EEG files directly into Google Drive without using local network bandwidth.

In [ ]:
# Automatic MOABB-based downloader for standardized BCI cohorts
if DATASET == "cho2017":
    from moabb.datasets import Cho2017
    moabb_ds = Cho2017()
    print("Streaming Cho2017 files directly to cloud storage...")
    # Pre-fetch subjects 1 to 5 for initial cohort run
    moabb_ds.download(subject_list=list(range(1, 6)))
elif DATASET == "bnci2014_001":
    from moabb.datasets import BNCI2014001
    moabb_ds = BNCI2014001()
    moabb_ds.download(subject_list=list(range(1, 10)))
elif DATASET == "bnci2014_004":
    from moabb.datasets import BNCI2014004
    moabb_ds = BNCI2014004()
    moabb_ds.download(subject_list=list(range(1, 10)))
else:
    print(f"Dataset {DATASET} configured for native adapter streaming from {DATA_DIR}")

## 4. Execute Unified Benchmark Pipeline
Computes Level 1 (fold), Level 2 (participant), and Level 3 (dataset) contrasts.

In [ ]:
# Run unified benchmark engine
!python scripts/run_unified_benchmark.py \
    --dataset {DATASET} \
    --data-dir {DATA_DIR} \
    --outdir {OUT_DIR} \
    --decoders csp_lda,ts_lr \
    --designs D3,D1s,P1 \
    --subjects 1-10

## 5. Inspect Results & Sealed Cryptographic Manifest

In [ ]:
import pandas as pd
import json

# Display Level 3 contrast summary
contrasts_path = f"{OUT_DIR}/design_comparisons.csv"
if os.path.exists(contrasts_path):
    df_comp = pd.read_csv(contrasts_path)
    display(df_comp)

# Display execution manifest
manifest_path = f"{OUT_DIR}/benchmark_manifest.json"
if os.path.exists(manifest_path):
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
    print(json.dumps(manifest, indent=2))

# Print cryptographic hashes of generated artifacts
!sha256sum {OUT_DIR}/*.csv